# 01 · Align clips, joints and source folds


A useful predictor comparison needs a fixed boundary between past inputs and
the teacher target. All clips from a source video stay together in five outer
folds. Three inner source folds choose settings within each outer-training set.
A source split is not proof that participants differ across recordings.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../docs/studies/future-innovation/README.md) ·
[Historical direct-v2 specification](../../docs/studies/future-innovation/direct-gate-protocol.md) ·
[Calibrated direct-v3 specification](../../docs/studies/future-innovation/direct-v3-repair-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation-direct-v3-dev-20260911")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Original runs build candidates and aligned poses. Cached direct-v3 verifies and reuses the frozen parent cohort and pose evidence. This can take hours. The existing stages verify and reuse completed work; inspect the resulting exclusions and overlays below.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    stage_error = attempt_stage("build-cohort", RUN_ROOT)
    if stage_error is None:
        stage_error = attempt_stage("extract-poses", RUN_ROOT)

The existing 50 clips and their source labels are reused unchanged in direct-v3.
The original cohort has 43 sources and 39–41 training clips per outer fold.
These sources have been inspected during debugging and are development data.
No clip is removed because a held-out joint becomes missing or a transform
produces an unusually difficult prediction.

| Quantity | Boundary and meaning |
|---|---|
| Prefix RGB, pose and normalization | frames 0–31 only |
| Skeleton history | 32 frames × 33 joints × x, y, confidence, validity |
| Teacher target region | frames 38–39 |
| Teacher target encoding context | full 64-frame clip, including later observations |
| Missing joint | no invented coordinate or velocity observation |
| Velocity summary | adjacent valid endpoints within each ordered 8-frame bin |

In [ ]:
from gavd6_sjepa.research_directions.future_innovation.fi_contracts import FRAME
fig, ax = plt.subplots(figsize=(9,2.2))
ax.broken_barh([(0,32)],(0,0.7),facecolors='#2f6f99',label='Predictor inputs')
ax.broken_barh([(38,2)],(0,0.7),facecolors='#d4803f',label='Target region')
ax.broken_barh([(0,64)],(1,0.5),facecolors='#bdd5c8',label='Full teacher context')
ax.set(xlim=(0,64),xticks=[0,8,16,24,32,38,40,64],yticks=[],xlabel='Clip frame boundary')
ax.legend(loc='upper center',bbox_to_anchor=(.5,1.4),ncol=3); plt.show()
if MODE != "teach":
    cohort = read_optional_table(RUN_ROOT,'manifests/gate-windows.csv')
    if cohort is not None:
        display(cohort.groupby('outer_fold').agg(test_clips=('window_id','size'),test_sources=('video_id','nunique'))
                .assign(training_clips=lambda t: len(cohort)-t.test_clips))
        print(f'{len(cohort)} clips from {cohort.video_id.nunique()} source videos; inspected development cohort.')
        columns=[c for c in ['window_id','video_id','outer_fold','source_first_frame','source_last_frame',
                             'context_pose_coverage','minimum_person_crop_retention','horizon_seconds'] if c in cohort]
        display(cohort[columns])
    else:
        print('No local frozen cohort. Source availability and cohort eligibility are separate stages.')

In [ ]:
if MODE != "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    run_path=RUN_ROOT/'config/run-contract.json'
    if run_path.is_file() and read_json(run_path).get('protocol')=='direct-v3':
        from gavd6_sjepa.research_directions.future_innovation.fi_cache_reuse import load_reused_cache
        cohort, arrays=load_reused_cache(RUN_ROOT)  # Verified and read-only; no raw media.
        sk=arrays['skeleton']
        display(pd.DataFrame({'window_id':cohort.window_id,'valid_fraction':sk[...,3].mean(axis=(1,2)),
                              'confidence_mean':sk[...,2].mean(axis=(1,2)),
                              'missing_fraction':1-sk[...,3].mean(axis=(1,2))}))
        fig,ax=plt.subplots(figsize=(9,3))
        ax.imshow(sk[0,...,3].T,aspect='auto',vmin=0,vmax=1,cmap='Blues')
        ax.set(xlabel='Prefix frame',ylabel='Joint',title=f'Validity: {cohort.window_id.iloc[0]}'); plt.show()
        print('Inherited cache integrity checked here. Raw alignment evidence is reused from parent receipts.')

In [ ]:
if MODE == "execute":
    require_stage_success(stage_error)

## What this step establishes

The same cohort and source folds now define every fitting boundary. Next establish how teacher features were encoded and which validity evidence is reused.

Continue with [02_teacher_features_and_validity.ipynb](02_teacher_features_and_validity.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")